In [ ]:
# Setup — run this FIRST (restart the kernel if you already imported shipit_agent).
#
# This notebook lives in notebooks/computer_use/, so the repo root is TWO levels
# up (parent.parent). Put it ahead of site-packages BEFORE the first import so
# the LOCAL checkout wins — otherwise Jupyter loads the installed shipit_agent,
# which lags this branch (e.g. the new computer_use list_apps/focus_app actions).
import sys
import pathlib

repo = pathlib.Path.cwd().parent.parent          # notebooks/computer_use -> repo root
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import shipit_agent
print(f"shipit-agent {shipit_agent.__version__}")
print(f"from {shipit_agent.__file__}")
assert "site-packages" not in shipit_agent.__file__, (
    "Still importing the installed package — restart the kernel and run this cell first."
)

In [ ]:
import importlib.util
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

from shipit_agent import Agent, FunctionTool, format_event_line
from shipit_agent.llms import LLMResponse, LiteLLMChatLLM
from shipit_agent.mcp import MCPStreamableHTTPTransport, RemoteMCPServer
from shipit_agent.models import ToolCall
from shipit_agent.tools import ToolContext

print('Python:', sys.version.split()[0])
print('Workspace:', Path.cwd())

In [ ]:
VKEY = "/Users/rahulraj/Documents/VivaDrive/Notebook/fleet-agent/fleetflow-477117-bcc775f027a3.json"

provider, model = "vertex", "gemini-2.5-flash"


if provider == "bedrock":
    os.environ.setdefault("AWS_PROFILE", "default")
    os.environ.setdefault("AWS_REGION_NAME", "us-east-1")
    os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
elif provider == "vertex":
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = VKEY
    os.environ["VERTEXAI_PROJECT"] = json.load(open(VKEY))["project_id"]
    os.environ["VERTEXAI_LOCATION"] = "us-central1"


In [ ]:

from shipit_agent import Agent
from shipit_agent.llms.factory import build_llm_from_settings

settings = {"provider": provider, "model": model}
if provider == "litellm":
    settings["SHIPIT_LITELLM_MODEL"] = model

llm = build_llm_from_settings(dict(settings))
llm

In [ ]:
# Real computer_use demo — Vertex Gemini DRIVES the desktop tool, live.
#
# The model calls computer_use itself: list_apps to see what's running, then a
# real screenshot that is fed straight back to Gemini (vision), which then
# describes YOUR actual screen. Safe by construction: the prompt only allows
# list_apps + screenshot (no clicking/typing/dragging).
import time
from shipit_agent import Agent
from shipit_agent.tools.computer_use.computer_use_tool import ComputerUseTool

cu = ComputerUseTool()
print("computer_use actions available:",
      ", ".join(cu.schema()["function"]["parameters"]["properties"]["action"]["enum"]))

agent = Agent(
    llm=llm,
    tools=[cu],        
    deferred_tools=True,               # just the desktop tool, so it's clearly used
    auto_use_skills=False, 
    auto_project_memory=False, 
    skill_source=None,
    max_iterations=6,
)

task = (
    "Look at my screen with the computer_use tool. "
    "First call computer_use action='list_apps' to see the running applications. "
    "Then call computer_use action='screenshot' to capture the screen. "
    "Then describe what you actually see in 2-3 sentences. "
    "IMPORTANT: only list_apps and screenshot — do NOT click, type, drag or press keys."
)
print(f"\nTASK: {task}\n\n--- live (real screen + real Gemini) ---\n", flush=True)

answer, saw_image, events = "", False, []
for ev in agent.stream(task):
    events.append(ev)
    p = ev.payload or {}
    if ev.type == "tool_called":
        act = (p.get("arguments") or {}).get("action", "?")
        print(f"  → computer_use(action={act})", flush=True)
    elif ev.type == "tool_completed":
        md = p.get("metadata") or {}
        if md.get("apps"):
            print(f"    ✓ running apps: {md['apps'][:6]}", flush=True)
        if md.get("image_base64"):
            saw_image = True
            print(f"    ✓ screenshot captured ({len(md['image_base64'])} b64 chars) → fed to Gemini", flush=True)
    elif ev.type == "final_answer":
        answer = p.get("content", "") or answer

print(f"\nscreenshot reached the model: {saw_image}")
print(f"\nGemini's description of your real screen:\n{answer}")

In [ ]:
for ev in events:
    print(ev, flush=True)